# KMeans Clustering

## Clustering

Clustering is a branch of unsupervised machine learning models that seeks to learn from the properties of the data by identifying groups or clusters in the dataset. Clustering is different from classification as it seeks to identify a set of similar data points and calling the resulting set a cluster. Selecting an appropriate clustering algorithm is often difficult due to the number of choices available. For a comprehensive survey of clustering algorithms, refer [here](https://link.springer.com/content/pdf/10.1007/s40745-015-0040-1.pdf).The three most popular categories of clustering algorithms are


|Categories|Clustering Algorithm| 
|----|------|
|Partitional clustering|divides data objects into non overlapping groups| 
|Hierarchical clustering|assignments by building a hierarchy|
|Density-based clustering|assignments based on the density of data points in a region|

## K-Means Algorithm

The k-means algorithm searches for a predetermined number of clusters within an unlabeled dataset and is based on the assumptions that the optimal cluster will have

- “cluster center” which is the mean of all the points belonging to the cluster.
- each point is closer to its own cluster center than to other cluster centers.

Conventional k-means algorithm requires only a few steps and follows a two-step process called **Expectation-Maximization** approach. This consists of

1. Randomly selecting k-centroids or cluster center 
2. Repeat until converged
   - E-Step: assign points to the nearest cluster center
   - M-Step: set the cluster center to the mean

The "E-Step" or "Expectation Step" assigns each data point to its nearest centroid. Then, the "M-Step" or Maximization Step" computes the mean of all the points for each cluster and sets the new centroid. The k-means algorithms seeks to find 
k clusters within a data set. The clusters are chosen to reduce the inertia, the objective function

where $\mu_k$ is centroid of a cluster and $|C_k|$ is the number of points in cluster k

The cluster assignments are done based on squared Euclidean distance and algorithm tries to minimize the objective function. This process is repeated, iteratively, until the sum of the distance is minimized.

In [ ]:
from PIL import Image

import os
dir_path = r"C:\Old SSD Data\C Drive\Desktop\CQF Course\Python Labs\K-Means Clustering & Self Organizing Maps"

In [ ]:
Image.open(os.path.join(dir_path, 'kmean_1.png'))

### Import Libraries  : Exhaustive list, not all being used

In [ ]:
# data manipuation
import pandas as pd
import numpy as np

# panda technical analysis 
import pandas_ta as ta

# to reload another python script 
from importlib import reload

# data visualization
import matplotlib as mpl

import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10,6)
plt.style.use ='seaborn'
plt.rcParams['font.size'] = 8

# statistical visual representation
import seaborn as sns

import cufflinks as cf                                                        # new package - cufflinks
cf.set_config_file(offline=True, theme='pearl',dimensions=((1000,600)))       # cf.set_config() -- (w,h)

import plotly.express as px
px.defaults.width, px.defaults.height = 1000,600

# Self organizing maps(SOM)
from minisom import MiniSom

# K-means clustering(KMC)
from sklearn.cluster import KMeans

# scikit learn(sklearn) package is used primarily for machine learning
# always remember to perform pre-processing and model selcetion before using model implementation
import sklearn
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.model_selection import train_test_split,TimeSeriesSplit,cross_val_score,GridSearchCV,RandomizedSearchCV

from sklearn.pipeline import Pipeline

# sklearn evaluation metrics
from sklearn.metrics import precision_recall_curve, roc_curve,RocCurveDisplay,ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, f1_score, recall_score,precision_score,roc_auc_score,auc
from sklearn.metrics import classification_report, confusion_matrix

# Support Vector Machine(SVM)
from sklearn.svm import SVC

# DTEM(Decision Trees and Ensemble Models) - Bagging - Random Forest
from sklearn.ensemble import RandomForestClassifier

# DTEM(Decision Trees and Ensemble Models) - Boosting - Gradient Boosting(Extreme Gradient Boosting)
from xgboost import XGBClassifier,plot_importance,to_graphviz

# K nearest neighbours(KNN)
from sklearn.neighbors import KNeighborsClassifier

# Principal Component Analysis(PCA)
from sklearn.decomposition import PCA

# statistical analysis & optimization
import scipy.stats as scs
import scipy.optimize as sco

# statsmodels is a Python package that provides a complement to scipy for statistical computations
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Shapley values, used in DTEM
import shap

# Ignore warnings - optional
import warnings
warnings.filterwarnings('ignore')

# data retrival
import yfinance as yf
 
# float display
pd.options.display.float_format = '{:0.4f}'.format

# operating system
import os

# Image package
from PIL import Image

# table formation
from tabulate import tabulate

# datetime handling
from datetime import datetime

## Implementation

In [ ]:
# Read components of dow from Yahoo Finance

dow = pd.read_html('https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average')       # pd.read_html()
dow_stocks = list(dow[1]['Symbol'])
dow_company_names = list(dow[1]['Company'])

dow_ohlc = {symbol:yf.download(tickers = symbol, start='2010-01-01', end='2022-05-31',progress =False)  for 
            symbol in dow_stocks }

In [ ]:
for symbol in dow_stocks:
    high = dow_ohlc[symbol]['High']
    low = dow_ohlc[symbol]['Low']
    close = dow_ohlc[symbol]['Close']
    dow_ohlc[symbol]['ATR'] = ta.atr(high = high, close=close,low = low)

### Stock Return

In [ ]:
stock_price =  pd.DataFrame({symbol: dow_ohlc[symbol]['Close'] for symbol in dow_stocks})

stock_ret = stock_price.resample('W-Fri').last().pct_change()

In [ ]:
# Check for missing values
# it shows 481 values missing for DOW stock so better to drop it, for rest we can do forward filling
# since these are numeric columns hence we uses np.isnan() istead of isna()
np.isnan(stock_ret).sum()

In [ ]:
# Fill forward the missing values(Still something missing drop it such as first row)and drop DOW form the list 
stock_ret.drop(columns = 'DOW', inplace = True)
stock_ret.fillna(method = 'ffill',inplace = True)
stock_ret.dropna(inplace = True)

In [ ]:
np.isnan(stock_ret).sum()

### Stocks ATR

In [ ]:
# ATR - Average true range
stock_atr =  pd.DataFrame({symbol: dow_ohlc[symbol]['ATR'] for symbol in dow_stocks})
stock_atr = stock_atr.resample('W-Fri').mean()

In [ ]:
# Fill backward the missing values and drop DOW form the list
stock_atr.fillna(method='bfill', axis=0, inplace=True)
stock_atr.drop(['DOW'], axis=1, inplace=True)

weekly_atr = stock_atr.T                          # required for K-Mean clustering

In [ ]:
np.isnan(stock_ret).sum()

In [ ]:
# Plot weekly atr values for MMM
stock_atr.iloc[:,0].plot()
plt.show()

### Elbow Plot

The number of clusters is a hyperparameter to clustering models and choose the optimal number of clusters is critical for the model. If every point is its own cluster, the inertia is zero and hence choosing the number of clusters that yields the lowest results in not appropriate. Instead, we need to identify the inflection point where increasing the number of clusters no longer result in significant drop in inertia. Elbow plot helps us to select the optimal number of clusters.

inertia is kind of measuring error(SSE)

In [ ]:
n_clusters = range(2,30)
inertia = []

for n in n_clusters:
    kmeans = KMeans(n_clusters = n)
    kmeans.fit(X = weekly_atr)                 # X : {array-like, sparse matrix} of shape (n_samples, n_features)
    inertia.append(kmeans.inertia_)
    

# here we are considering 5 % relative inertia as a cutoff to selct no of clusters
# if no of clusters many ---> high bias, low variance
# if no of clusters less ---> low bias, high variance

plt.plot(n_clusters,np.divide(inertia,inertia[0]))
plt.hlines(y=0.1,xmin= n_clusters[0], xmax = n_clusters[-1],colors = 'red',linestyle = '--')
plt.hlines(y=0.05,xmin= n_clusters[0], xmax = n_clusters[-1],colors = 'green',linestyle = '--')
plt.xlabel('clusters')
plt.ylabel('relative inertia')
plt.legend(['inertia', '10% relative inertia', '5% relative inertia']);
plt.show()

### Silhouette Coefficient

Silhouette Coefficient is another metric measure of how dense and separated are the clusters. Silhouette coefficient values range between -1 and 1. Larger numbers indicate that samples are closer to their clusters than they are to other clusters. Silhouette coefficient implementation is straight forward where the average silhouette coefficient of all the samples is summarized into one score. The silhouette_score function needs a minimum of two clusters, or it will raise an exception.

### Model Pre-implementation & Model Implementation

We will now fit the cluster model. Given the timeseries characteristics, the dataset needs to be normalized for the model to work properly. The normalizing can be accomplished by using the Normalizer transformer from Scikit-Learn

In [ ]:
#---------------------------------------------------------------------------------------------------------------
# Initialize the model --> Pipeline --> fit() --> predict()
#---------------------------------------------------------------------------------------------------------------

#---------------------------------------------------------------------------------------------------------------
# Hyperparameters : n_clusters
#---------------------------------------------------------------------------------------------------------------

# pipeline - Sequentially apply a list of transforms and a final estimator.Intermediate steps of the pipeline must be 
# 'transforms', that is, they must implement `fit` and `transform` methods.

# KMeans(n_clusters = 15) ===> initialize the model

# Pipleine contains both model pre-implementation and model implementation(initialize the model)
model= Pipeline([('sclaer',MinMaxScaler()),('cluster',KMeans(n_clusters = 15))])     # Pipeline()

model.fit(X = weekly_atr)                                                            # model.fit()

cluster_labels = model.predict(weekly_atr)                                           # model.predict()

### Model attributes

In [ ]:
model.intertia_                                        # model.inertia_
model.get_params()                                     # model.get_params()     
model['cluster'].cluster_centers_                      # model['cluster'].cluster_centers_
model['cluster'].cluster_centers_.shape                # model['cluster'].cluster_centers_.shape : shape(n_clusters,n_features)

### Summary of Results

In [ ]:
companies = stock_atr.columns
companies_names = dow_company_names
companies_names.remove('Dow')

In [ ]:
df = pd.DataFrame({'Cluster':cluster_labels,
                   'Company':companies,
                   'Company Name':companies_names,
                   'ATR':stock_atr.mean(axis=0),
                   'Avg Weekly Return':stock_ret.mean(axis=0)})

df.sort_values(by= 'Cluster',ignore_index=True,inplace = True)

Thus, using k-means, we have discovered companies that exhibit similar volatility characteristics as measured by the Average True Range (ATR).

In [ ]:
plt.scatter(x = df['Company'], y= df['Cluster'])
plt.ylabel('Cluster')
plt.title('Dow Stocks ATR Clustering');
plt.show()

In [ ]:
df.to_csv('cluster_new.csv')

## References

[K Means](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)

[Silhoutte Score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)

[Python Resources](https://github.com/kannansingaravelu/PythonResources)

# Self Organizing Maps

Self-organizing maps (SOM) are a class of unsupervised learning neural networks used for feature detection. SOM are used to project a high dimensional data into a smaller set and are used for dimensionality reduction.

SOMs differ from conventional artificial neural networks as they apply competitive learning when compared to error correlated learning, which involves backpropagation and gradient descent. In SOM, the training data has no labels and the map learns to differentiate and distinguish features based on similarities.

## SOM Algorithm

Step 1: Initialize weights $w_{ij}$ <br>
Step 2: Calculate distance $d_{ij} = \sum (w_{ij} - x_i)^2$ <br>
Step 3: Calcualte $w_{ij}(t+1) =w_{ij}(t) + \theta(t).\alpha(t).(x_i(t) - w_{ij}(t))$

where $x_i(t)$ is the characteristics of the samples, $\alpha(t)$ is a learning rate and $\theta(t)$ is the neighbourhood function.

The weights are adjusted iteratively from the Euclidean distance by updating the learning rate. The neuron whose weight vector is most similar to the input is called the best matching unit (BMU). Once the BMU has been determined, we then calculate which of the other nodes are within the BMU’s neighborhood. Nodes found within the radius are deemed to be inside the neighborhood of the BMU. This process is repeated until convergence or number of iterations.

In [ ]:
df = pd.read_csv('cluster_new.csv', index_col=0)

## Define Feature Matrix

In [ ]:
X = df[['ATR','Avg Weekly Return']]             # feature matrix

## Model Pre-implementation

In [ ]:
scaler = MinMaxScaler()  
X = scaler.fit_transform(X)            # scaler.fit_transform()

## SOM Implementation

In [ ]:
#---------------------------------------------------------------------------------------------------------------
# Initialize the model --> random weight assign --> train batch
#---------------------------------------------------------------------------------------------------------------

#---------------------------------------------------------------------------------------------------------------
# Hyperparameters :x,y,input_len,learning_rate,sigma
#---------------------------------------------------------------------------------------------------------------

# Initialize a 5x5 SOM - Thumb rule is 5 * sqrt(n) > 5x5 SOM, n= n_samples which is 29 in this case
# totoal clusters = 25 , also calld neurons
som = MiniSom(x =5,y=5, input_len =2,learning_rate=0.5, sigma=5, random_seed=42 )        # Initializes a Self Organizing Maps

# Initialize with randon weights

som.random_weights_init(X)                                                               # som.random_weights_init()

# Trains SOM with 10000 iterations

som.train_batch(X,num_iteration= 10000, verbose=True)                                                   # som.train_batch()

## Model attributes

In [ ]:
som.winner(X[1])                       # som.winner(X[ix]) - for ix sample winning cluster
som.distance_map()                     # som.distance_map() - weight matrix - Wij

In [ ]:
# SOM Plot

for ix in range(len(X)):
    winner = som.winner(X[ix])
    winner = som.winner(X[ix])
    plt.text(x = winner[0], y= winner[1],s = df.Company[ix],bbox = dict(facecolor='white', alpha=0.5, lw=0))
plt.imshow(som.distance_map())   
plt.grid(False)
plt.colorbar()
plt.show() 

In [ ]:
# SOM table

# List the corresponding companies
for ix in range(len(X)):
    winner = som.winner(X[ix])
    print(winner[0],winner[1], df['Company Name'].iloc[ix])

## References

[MiniSom](https://github.com/JustGlowing/minisom)

[Python Resources](https://github.com/kannansingaravelu/PythonResources)